## Aeropulse — Gold: Destination Airport Dimension

**Purpose:** Builds `dim_destination_airport` directly from silver `flight` — deduplicated on `destination_airport_code` — rather than from silver `airport`, since this dimension only needs the destination code/city pair that `fact-flight` joins back to.

**Batch parameters:** `batch_id`, `batch_year`

**Depends on:** `gold-environment`, `gold-helper` (run via `%run`)

**Reads:** `silver.flight` (filtered to `batch_id`)

**Writes:** `dim_destination_airport` (merge on `airport_destination_sk`)


In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 3, Finished, Available, Finished, False)

In [2]:
batch_id = ""
batch_year = ""

StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 4, Finished, Available, Finished, False)

In [3]:
%run gold-environment

StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 5, Finished, Available, Finished, True)

In [4]:
%run gold-helper

StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 8, Finished, Available, Finished, True)

## read silver flight dataframe

In [5]:
flight_df = spark.read.format('delta').load(flight_silver_path).filter(F.col("batch_id")==batch_id)

StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 9, Finished, Available, Finished, False)

## creating dim-airport-destination df

In [6]:
## deduplicating flights_df by airport destination attributes

airport_dest_lookup = flight_df.select(
    F.col("destination_airport_code"),
    F.col("destination_city_name"),
).dropDuplicates(["destination_airport_code"])

StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 10, Finished, Available, Finished, False)

## generate a surrogate key

In [7]:
# create surrogate key
airport_dest_lookup = add_sk_key(airport_dest_lookup, ["destination_airport_code"], "airport_destination_sk")


StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 11, Finished, Available, Finished, False)

## write to gold layer

In [8]:
# write to gold layer

update_cols = [c for c in airport_dest_lookup.columns if c not in ["airport_destination_sk"]]

write_to_gold(
    airport_dest_lookup,
    "dim_destination_airport",
    "s.airport_destination_sk = t.airport_destination_sk",
    update_cols
)


StatementMeta(, f63efc1e-1a9c-4a2d-baca-ff1cf7d848dc, 12, Finished, Available, Finished, False)